# 04 - Player features

**Input:** `data/processed/strokes_all.csv`, `data/raw/match.csv`
**What it does:** builds a per-player feature vector describing (1) shot mix, (2) spatial behavior (depth from court center, mean displacement applied), (3) rally-shape behavior (rally length when winning/losing, win rate), (4) win-reason style (share of wins by own winner vs forced error), and (5) landing-location variance. These features feed the clustering in notebook 05.
**Output:** `data/processed/player_features.csv` (one row per player).

Uses `map_player_names` from `utils.py` to resolve the per-match A/B codes to real names (shared with notebooks 06-11).

In [1]:
## --- path bootstrap: run from the repo root no matter where this is launched ---
## nbconvert and some editors set the working directory to the notebook's own
## folder. Walk up until we find the repo root (the folder containing data/),
## chdir there so relative data paths resolve, and put code/ on sys.path so the
## shared modules (utils.py, shot_translations.py) import cleanly.
import os, sys
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
if os.path.join(_d, "code") not in sys.path:
    sys.path.insert(0, os.path.join(_d, "code"))
print("working directory:", os.getcwd())

working directory: /Users/aakankshvaidya/Desktop/qss20_final_project


In [2]:
import os
import numpy as np
import pandas as pd

import utils

In [3]:
## CONFIG
PROC_DIR = utils.PROC_DIR
OUT_PATH = os.path.join(PROC_DIR, "player_features.csv")

## shot-category groupings: collapse the 18 shot types into broad tactical buckets
SHOT_CATEGORIES = {
    "smash":       ["smash"],
    "wrist_smash": ["wrist smash"],
    "attacking":   ["drive", "rush", "back-court drive", "driven flight"],
    "neutral":     ["clear", "push", "drop", "passive drop"],
    "defensive":   ["lob", "defensive return lob", "defensive return drive"],
    "net":         ["net shot", "return net", "cross-court net shot"],
    "serve":       ["short service", "long service"],
}

def assign_category(shot_type):
    ## map a shot type to its broad category; anything else -> 'other'
    for cat, members in SHOT_CATEGORIES.items():
        if shot_type in members:
            return cat
    return "other"

## Load data and resolve player names\nThe A/B -> name mapping (and its before/after merge diagnostics) lives in `utils.map_player_names`.

In [4]:
strokes = utils.load_strokes()
matches = utils.load_matches()
print("strokes:", strokes.shape, " matches:", matches.shape)
strokes = utils.map_player_names(strokes, matches)

strokes: (52356, 31)  matches: (58, 13)
map_player_names: rows before merge = 52356, after = 52356
map_player_names: unique players resolved = 35


## Shot-mix features\nProportion of each player's shots in each broad category.

In [5]:
strokes["shot_category"] = strokes["shot_type"].apply(assign_category)
shot_mix = (
    strokes.groupby(["player_name", "shot_category"]).size().unstack(fill_value=0)
)
shot_mix = shot_mix.div(shot_mix.sum(axis=1), axis=0)
shot_mix.columns = [f"pct_{c}" for c in shot_mix.columns]
print("shape:", shot_mix.shape)
print(shot_mix.head(3))

shape: (35, 8)
                  pct_attacking  pct_defensive   pct_net  pct_neutral  \
player_name                                                             
Aakarshi KASHYAP       0.027094       0.120690  0.224138     0.440887   
Akane YAMAGUCHI        0.024201       0.175540  0.211036     0.346563   
An Se Young            0.034501       0.173275  0.227600     0.369722   

                  pct_other  pct_serve  pct_smash  pct_wrist_smash  
player_name                                                         
Aakarshi KASHYAP   0.002463   0.078818   0.078818         0.027094  
Akane YAMAGUCHI    0.019361   0.083253   0.094869         0.045176  
An Se Young        0.018795   0.080330   0.053296         0.042482  


## Spatial features\nWithout a per-stroke side flag, absolute court-y is ambiguous across players, so we use distance from the court's center line (`depth_from_center`) plus mean displacement applied.

In [6]:
court_center_y = strokes["player_location_y"].median()
print("estimated court center y:", court_center_y)
strokes["depth_from_center"] = (strokes["player_location_y"] - court_center_y).abs()

spatial = strokes.groupby("player_name").agg(
    mean_depth_from_center=("depth_from_center", "mean"),
    mean_displacement_applied=("displacement", "mean"),
).reset_index().set_index("player_name")
print(spatial.head(3))

estimated court center y: 456.0
                  mean_depth_from_center  mean_displacement_applied
player_name                                                        
Aakarshi KASHYAP              126.288177                 191.738532
Akane YAMAGUCHI               100.678244                 154.243676
An Se Young                    86.727083                 154.550443


## Rally-shape features\nOne row per (player, rally) so a player counts once per rally; then mean rally length when winning vs losing, and overall rally win rate.

In [7]:
rally_player = (
    strokes.dropna(subset=["rally_winner"])
    .groupby(["match_id", "set_num", "rally", "player_name"])
    .agg(hitter_won_rally=("hitter_won_rally", "first"),
         rally_length=("ball_round", "max"))
    .reset_index()
)
won = rally_player[rally_player["hitter_won_rally"] == True]
lost = rally_player[rally_player["hitter_won_rally"] == False]
mean_len_won = won.groupby("player_name")["rally_length"].mean().rename("mean_rally_length_when_won")
mean_len_lost = lost.groupby("player_name")["rally_length"].mean().rename("mean_rally_length_when_lost")
win_rate = rally_player.groupby("player_name")["hitter_won_rally"].mean().rename("rally_win_rate")
rally_shape = pd.concat([mean_len_won, mean_len_lost, win_rate], axis=1)
print(rally_shape.head(3))

                  mean_rally_length_when_won  mean_rally_length_when_lost  \
player_name                                                                 
Aakarshi KASHYAP                   10.806452                     9.212766   
Akane YAMAGUCHI                    10.517241                    11.386179   
An Se Young                        10.393443                     9.892265   

                  rally_win_rate  
player_name                       
Aakarshi KASHYAP        0.397436  
Akane YAMAGUCHI         0.541045  
An Se Young             0.502747  


## Win-reason features\nShare of a player's rally wins that ended on their own winning shot (落地致勝) vs an opponent error. High = offensive finisher; low = error-baiter.

In [8]:
rally_end = (
    strokes.dropna(subset=["win_reason"])
    .groupby(["match_id", "set_num", "rally"])
    .agg(rally_winner_name=("player_name", "first"), win_reason=("win_reason", "first"))
    .reset_index()
)
rally_end["was_winner_shot"] = rally_end["win_reason"] == "落地致勝"
win_style = (
    rally_end.groupby("rally_winner_name")["was_winner_shot"].mean().rename("pct_wins_by_winner")
)
win_style.index.name = "player_name"
print(win_style.describe().round(3))

count    35.000
mean      0.302
std       0.056
min       0.150
25%       0.270
50%       0.302
75%       0.347
max       0.418
Name: pct_wins_by_winner, dtype: float64


## Landing-location variance\nHow broadly a player distributes shot landing spots (std of landing_x / landing_y).

In [9]:
landing_var = (
    strokes.dropna(subset=["landing_x", "landing_y"])
    .groupby("player_name")
    .agg(landing_x_std=("landing_x", "std"), landing_y_std=("landing_y", "std"))
)
print(landing_var.describe().round(2))

       landing_x_std  landing_y_std
count          35.00          35.00
mean          155.03         103.44
std            16.23          14.72
min           123.99          70.22
25%           145.65          96.45
50%           152.96         103.10
75%           162.35         112.77
max           202.41         131.36


## Combine all features and save

In [10]:
features = shot_mix.join(spatial, how="outer").join(rally_shape, how="outer")
features = features.join(win_style, how="left").join(landing_var, how="left")

## drop the tiny 'other' shot bucket unless some player exceeds 5%
if "pct_other" in features.columns:
    if features["pct_other"].max() > 0.05:
        print("keeping pct_other (some player exceeds 5%)")
    else:
        features = features.drop(columns="pct_other")
        print("dropping pct_other (all players <5%)")

## attach stroke counts for later reliability diagnostics
strokes_per_player = strokes.groupby("player_name").size().rename("n_strokes")
features = features.join(strokes_per_player)

print("final feature matrix shape:", features.shape)
print("columns:", list(features.columns))
print("strokes per player (min/median/max):",
      features["n_strokes"].min(), features["n_strokes"].median(), features["n_strokes"].max())

os.makedirs(PROC_DIR, exist_ok=True)
features.to_csv(OUT_PATH)
print("saved:", OUT_PATH, " rows (players):", len(features))

dropping pct_other (all players <5%)
final feature matrix shape: (35, 16)
columns: ['pct_attacking', 'pct_defensive', 'pct_net', 'pct_neutral', 'pct_serve', 'pct_smash', 'pct_wrist_smash', 'mean_depth_from_center', 'mean_displacement_applied', 'mean_rally_length_when_won', 'mean_rally_length_when_lost', 'rally_win_rate', 'pct_wins_by_winner', 'landing_x_std', 'landing_y_std', 'n_strokes']
strokes per player (min/median/max): 366 1073.0 4482
saved: data/processed/player_features.csv  rows (players): 35
